# Olist E-commerce Delivery Performance Analysis

**Business question:** Which order and fulfilment factors are associated with delivery delay? The target is actual delivery date minus estimated delivery date. Positive values mean late; negative values mean early.

## Results from the uploaded dataset
- Delivered orders analysed: **96,470**
- Late orders: **7,826 (8.11%)**
- On or before estimate: **88,644 (91.89%)**
- Mean delay: **-11.18 days**
- Median delay: **-11.95 days**
- Review-score / delay correlation: **-0.267**

The regression test R² is low. This is an important limitation, not a result to hide.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Load and check the prepared order-level dataset
Run this notebook from the project root folder.

In [ ]:
df = pd.read_csv('data/processed/delivered_orders_analysis.csv')
print('Shape:', df.shape)
print('Duplicate rows:', df.duplicated().sum())
print(df.isna().sum().sort_values(ascending=False).head(15))
df[['delivery_delay_days','approval_hours','item_count','total_price','total_freight']].describe()

## Exploratory data analysis
The project folder contains 12 saved visualisations. This cell recreates the delivery-delay distribution.

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df.delivery_delay_days.dropna(), bins=60)
plt.axvline(0, linestyle='--')
plt.xlabel('Delivery delay in days (positive = late)')
plt.ylabel('Orders')
plt.title('Delivery delay distribution')
plt.show()

## Regression models
An 80/20 train-test split is used with random_state=42. Review score is not a predictor because it is recorded after delivery. Model 2 adds interaction terms.

In [ ]:
cols=['delivery_delay_days','approval_hours','item_count','total_price','total_freight','seller_count','cross_state','purchase_month','purchase_weekday','customer_state','seller_state','main_category']
reg=df[cols].dropna()
train,test=train_test_split(reg,test_size=0.2,random_state=42)
base='delivery_delay_days ~ approval_hours + item_count + total_price + total_freight + seller_count + cross_state + purchase_month + purchase_weekday + C(customer_state) + C(seller_state) + C(main_category)'
m1=smf.ols(base,data=train).fit()
m2=smf.ols(base+' + total_freight:cross_state + approval_hours:item_count',data=train).fit()
for name,m in [('Main effects',m1),('Interactions',m2)]:
    pred=m.predict(test)
    print('\n',name)
    print('Train R2:',m.rsquared,'Adjusted R2:',m.rsquared_adj,'AIC:',m.aic,'BIC:',m.bic)
    print('Test R2:',r2_score(test.delivery_delay_days,pred),'RMSE:',mean_squared_error(test.delivery_delay_days,pred)**0.5,'MAE:',mean_absolute_error(test.delivery_delay_days,pred))

## Interpretation and limitations
The interaction model's test R² is only about 0.038, so the model explains a small share of delivery-delay variation. More information such as carrier, route distance, weather and warehouse capacity may be needed. These results show associations, not causes.

## SQL work
The SQLite database is `data/olist.sqlite`. Fourteen SQL queries are provided in `sql/queries.sql`, including joins, filters, aggregates and business-question queries.